In [1]:
from google.colab import drive
drive.mount('/content/drive') # Access Drive folder

Mounted at /content/drive


In [2]:
# List everything in the drive folder
!ls -la /content/drive/MyDrive/OCR_images/

total 11738
-rw------- 1 root root   112089 Jul 31 01:51 check_this_image_1.jpg
-rw------- 1 root root 10190856 Jul 31 22:28 kids_books_letters.mp4
-rw------- 1 root root    12861 Jul 28 00:11 paper-cash-sell-receipt-vector.jpg
-rw------- 1 root root   181352 Jul 28 00:36 wallmart-reciept-hands.jpg
-rw------- 1 root root   185819 Jul 28 00:17 walmart_2.png
-rw------- 1 root root    47283 Jul 27 00:54 walmart-receipt.png
-rw------- 1 root root   458428 Jul 28 01:01 wendys-reciept-2.jpg
-rw------- 1 root root    36455 Jul 28 00:45 wendys-reciept-3.jpg
-rw------- 1 root root   791657 Jul 28 00:44 wendys-reciept.jpg


In [3]:
!wget https://ollama.com/install.sh
!nvidia-smi

--2026-08-05 23:43:03--  https://ollama.com/install.sh
Resolving ollama.com (ollama.com)... 34.36.133.15
Connecting to ollama.com (ollama.com)|34.36.133.15|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: https://github.com/ollama/ollama/releases/latest/download/install.sh [following]
--2026-08-05 23:43:04--  https://github.com/ollama/ollama/releases/latest/download/install.sh
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/ollama/ollama/releases/download/v0.32.6/install.sh [following]
--2026-08-05 23:43:04--  https://github.com/ollama/ollama/releases/download/v0.32.6/install.sh
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/658928958/a6901d58-bb0e-48e6-b59a-61

In [4]:
import os

# Set the system path environment variable so Ollama can find the T4 libraries
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

!chmod +x install.sh
!sudo apt-get install zstd
!./install.sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (569 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently i

In [5]:
import subprocess
import time

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(5)
print("Ollama is up and running")

Ollama is up and running


In [6]:
!ollama pull qwen2.5vl:7b
!pip install ollama

In [7]:
# Sliding Window Approach
import cv2
import ollama
import tempfile
import os

video_path='/content/drive/MyDrive/OCR_images/kids_books_letters.mp4'
model_name = 'qwen2.5vl:7b'

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("The Video could not be opened")
    exit()

results_file="video_text_results.txt"
window_size_secs = 10 # Process 10 seconds of video at a time
current_second = 0
final_narrative = []
frame_paths = []
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_seconds = int(total_frames / fps)

print(f" Video Successfully loaded. \n Number of seconds:{total_seconds:.2f} \n Total Frames:{total_frames} \n FPS:{fps}")

while current_second < total_seconds:
    window_end = min(current_second + window_size_secs, total_seconds) # calculate limits for current 10 secs
    print(f'\n Current Window:{current_second}s to {window_end}s')

    # with tempfile.TemporaryDirectory() as temp_dir:
    with tempfile.TemporaryDirectory(dir='/content') as temp_dir: # Use in colab only
        print({temp_dir})
        for sec in range(current_second, window_end): # Extract one frame window per seconds for this window/10 seconds of video
            target_frame=sec * fps
            cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
            ret, frame = cap.read()

            if ret:
                path = os.path.join(temp_dir, f"img_second{sec}.png")
                cv2.imwrite(path, frame)
                frame_paths.append(path)

        if frame_paths:
            prompt = (
                f"You are watching a segment of a video from second {current_second} to {window_end}"
                "Extract all text, book titles, headlines, or captions that appear in the screen of the video"
                "Be brief and only extract full words or titles, avoid nonsense characters"
            )

            try:
                message=[{
                        'role':'user',
                        'content': prompt,
                        'images': frame_paths
                }]

                response=ollama.chat(
                    model=model_name,
                    messages=message,
                    options={
                        'num_ctx':  8192 # Increase the tokens so Ollama can get more images at the same time
                    }
                )

                result_text=response['message']['content'].strip()
                if result_text:
                    final_narrative.append(f"[{current_second}s - {window_end}s]\n{result_text}")
                    print(f"Extracted:\n{result_text}")
                    frame_paths.clear()

            except Exception as e:
                print(f"Error:{e}")

    current_second+=window_size_secs

cap.release()
print(final_narrative)
with open(results_file, mode='w', encoding='utf-8') as txt_file: # Writes the results in a text file
    for chunk in final_narrative:
        txt_file.write(chunk)
        txt_file.write("\n" + "-"*40 + "\n")

print(f"results file created: {results_file}")



 Video Successfully loaded. 
 Number of seconds:172.00 
 Total Frames:5178 
 FPS:30

 Current Window:0s to 10s
{'/content/tmps290aj7n'}
Extracted:
The Beatrix Potter Collection Volume Two

 Current Window:10s to 20s
{'/content/tmpriril8b2'}
Extracted:
THE TALE OF THE FLOPSY BUNNIES

 Current Window:20s to 30s
{'/content/tmptbrczeo_'}
Extracted:
SOMETIMES Peter Rabbit had no cabbages to spare.

 Current Window:30s to 40s
{'/content/tmp8zf3uln7'}
Extracted:
THE TALE OF THE FLOPSY BUNNIES BY BEATRIX POTTER

 Current Window:40s to 50s
{'/content/tmpvbslgfpw'}
Extracted:
The Beatrix Potter Collection Volume One

 Current Window:50s to 60s
{'/content/tmpirjy55cv'}
Extracted:
The Tale of Peter Rabbit

 Current Window:60s to 70s
{'/content/tmp8wd6eyt7'}
Extracted:
The Tale of Mr Tod begins on page 203

 Current Window:70s to 80s
{'/content/tmpsmn6p371'}
Extracted:
The Beatrix Potter Collection Volume Two

 Current Window:80s to 90s
{'/content/tmp63l4nwn5'}
Extracted:
They are available online 

In [ ]:
# !ls /content
print(final_narrative)